# 03 - Probes: seven pre-registered attempts to validate the vectors, and what survived

**What this notebook is for.** The second research question: do the emotion vectors *function*
as detectors? Give the model a scenario that implies an emotion without naming it ("my dog has
been missing for three days"); does the scenario's activation land closest to the right
emotion's probe vector, the way the source paper ([Anthropic's emotions paper](https://transformer-circuits.pub/2026/emotions), [arXiv 2604.07729](https://arxiv.org/abs/2604.07729)) demonstrates? Seven experiments answer this.
Every prediction was registered in TREE.md before scoring. The answer, under the readout
convention used throughout this notebook: no configuration passed the registered bar. What
survives is a coarse valence signal (rough good-vs-bad tone, not the specific emotion).

**Resolution, added 2026-07-22/23 (TREE Q1.H2.E9, claim C4).** After this campaign closed, an
audit-prompted change of readout convention passed the bar on BOTH models (`results/e9_centered_readout.json`):
centering the scenario activations (subtracting the scenario-set mean before taking cosines, the convention of the [ARENA evals curriculum](https://github.com/callummcdougall/ARENA_3.0)). That claim (C4) has since SURVIVED its falsify gate
(`results/falsify_c4_scorecard.json`, in the [experiment-artifacts dataset](https://huggingface.co/datasets/abotresol/emotion-vectors-experiment-artifacts) like every `results/` evidence file cited below): a selection-adjusted sweep-wide permutation null at
p at or below 0.0002, a random-probe-set control, a scenario bootstrap, and dialogue-probe
convergence. Probe lineage matters as much as readout: which model wrote the probe stories
sets how broadly probes pass (TREE Q1.H2.E10-E12; `notebooks/07_generator_lineages.ipynb`).
The sections below stand as the record of the uncentered-readout campaign. Its nulls remain
valid for that readout family. The bench notebooks in `archive/` hold every run in full.

**Key concepts.**
- *Models*: [google/gemma-4-31b](https://huggingface.co/google/gemma-4-31b) (base) and
  [google/gemma-4-31b-it](https://huggingface.co/google/gemma-4-31b-it) (instruction-tuned),
  the probed pair throughout.
- *Probe vector*: one direction per emotion, the mean activation over stories evoking that
  emotion. "Detection" means ranking all 12 probes by cosine against a scenario's activation.
- *Battery*: 12 scenarios, each implicitly evoking one emotion (the paper's Table 2, verbatim),
  plus 12 fresh held-out scenarios we wrote before any scoring, to catch selection effects.
- *Probe lineage*: unless a section says otherwise, probes come from the
  [`snae/emotion_stories_gemma_4_4B`](https://huggingface.co/datasets/snae/emotion_stories_gemma_4_4B)
  corpus, stories written by gemma-4-4B, a smaller external model. Section 3 introduces the second lineage: stories and dialogues written by the probed
  model itself (generator = probed model).
- *Pass bar (registered)*: the target emotion ranks in the top 3 of 12 probes for at least
  8 of 12 scenarios, on BOTH batteries.
- *Readout*: which token positions represent the scenario (last token, mean of all tokens,
  mean of content tokens).
- *Centering (the baseline a cosine is measured from)*: a probe only becomes a direction once
  a baseline is subtracted from it. Sections 1 and 2 subtract the mean of the full 171-word
  extraction (the "sweep convention"); sections 3 to 6 subtract the mean of the 12 battery
  probes, because those probe sets only ever extracted 12 emotions. Both are registered, and
  the package keeps them apart on purpose.
- *Neutral projection*: the paper's confound-removal step. PCA (principal component analysis)
  on neutral-text activations finds the directions that vary even without emotional content;
  the top ones are projected out of the probes.

**Index.**
1. The battery sweep: can any configuration detect the implied emotion? (instruct, then base)
2. Numerical intensity: does the reading move with what a number means? (paper Figure 3)
3. Probes from the model's own writing (self stories, dialogues)
4. The scale test: does more probe data help? (16 to 256 stories per emotion)
5. Probe-direction convergence: are the probes stable, or noisy?
6. Neutral projection: does the confound-removal step rescue detection?
7. Verdict and the standing explanation
8. Do the probes predict what the model would rather do? (paper Figure 4, left half)
9. Does steering with an emotion vector move those preferences? (paper Figure 4, right half)

Sections 8 and 9 were folded in from the paper-parity notebook. They are not an eighth and
ninth attempt at detection: they ask whether the same vectors predict and cause *behavior*,
which is the source paper's Figure 4, so they sit after the campaign's verdict rather than
inside it. The inventory of every source-paper figure and where it now lives is the front
matter of `notebooks/02_circumplex_geometry.ipynb`.

**The arc in one paragraph.** The base model failed the battery (archived runs 03-05,
reproduced at the end of section 1). The instruct model with its chat template failed it
(section 1). Probes rebuilt from the model's own stories and dialogues failed it (section 3).
Scaling the corpus 16x failed to move scores while making the probes provably stable
(sections 4-5). The paper's own confound projection improved scores by about one scenario
and failed to clear the bar (section 6). What does track is numerical-intensity direction
(section 2): the instruct model moves all 11 registered directions the right way, the base
model 7 of 11 (both printed in section 2). Geometry replicates; probe function does not,
*under the uncentered readout*, and does under the centered one (claim C4, gated).

**How the code is organised.** Every cell below is load-call-show over
`src/emotion_vectors/detection_report/`: it imports a function, calls it, and shows the
figure or the printed record. Two things stay in the cells on purpose, because a reader
should watch them being applied rather than trust a function: the registered pass rules, and
the asserts that reproduce archived results. Every input path resolves through
`emotion_vectors.artifacts.fetch`, so this notebook runs unchanged on any clone.


## 1. The battery sweep: can any configuration detect the implied emotion? (instruct model, then base)

The first and broadest attempt. If the probes detect an implied emotion at all, some choice of
layer, readout, and prompt format should do it. This section scores every one of them on both
models and grades the grid against the registered rule.


In [1]:
# load-call-show: resolve both models' probes and swept battery activations, score
# every (layer, readout, format) cell for the instruct model, and draw the grid
from emotion_vectors.detection_report import (
    BASE_LABEL,
    IT_LABEL,
    PASS_BAR,
    load_sweep_context,
    sweep_figure,
    sweep_grids,
)

context = load_sweep_context()
instruct = context.model(IT_LABEL)
instruct_grids = sweep_grids(instruct)
figure_instruct, stats_instruct = sweep_figure(
    instruct, instruct_grids, "our robustness extension of Anthropic Figure 2 (TREE Q1.H2.E4)"
)
figure_instruct.show()
print("\n".join(stats_instruct["lines"]))

gemma-4-31b-it, plain format, paper battery: best cell 4 of 12 (chance 3, bar 8)
gemma-4-31b-it, plain format, held-out battery: best cell 4 of 12 (chance 3, bar 8)
gemma-4-31b-it, chat format, paper battery: best cell 6 of 12 (chance 3, bar 8)
gemma-4-31b-it, chat format, held-out battery: best cell 4 of 12 (chance 3, bar 8)
best single combination across BOTH batteries: 4 of 12, bar 8


<details><summary><b>How to read this figure</b></summary>

**What the axes are.** Each row is a layer of the model (0 at the top, 57 at the bottom). Each
column is a readout: the rule for turning the prompt's token activations into a single vector
(last token, mean of all tokens, mean of content tokens). The two panel rows are the two prompt
formats: the scenario as plain text, and the same scenario wrapped in the model's chat
template. In each pair, the left panel holds the paper's 12 scenarios and the right panel our
12 held-out scenarios.

**What one cell is.** The number of scenarios, out of 12, whose true emotion ranked in the top
3 of the 12 probes. The color and the printed number say the same thing; the number is there
because a color scale cannot be read to one unit.

**The grading scale.** The colorbar is labelled at the two anchors that decide the result. A
cell at 3 is chance: ranking the target into the top 3 of 12 by luck happens about a quarter of
the time. A cell at 8 clears the registered bar, and any such cell is printed in bold. A good
result would be a bold cell in a left panel with a bold cell at the same layer and readout in
the right panel beside it. A bad result would be a grid sitting at chance everywhere. The
observed grid sits between the two and much nearer the bad end: the title reports the brightest
cell and the best combination scored on both batteries, both computed from the grids in this
cell, and no cell anywhere is bold.

**Valid readings.** "No combination reaches the registered bar." "What little signal there is
sits at the deepest layers." "Wrapping the prompt in the chat template does not rescue the
plain-format result."

**Invalid readings.** "The brightest cell detects the emotion." The brightest cell was chosen
by looking at a left panel, so it means nothing until it is confirmed at the same coordinates
on the right panel; the next cell applies exactly that rule. "Brighter is better." One step
above chance is one scenario, which a single lucky ranking produces.

</details>


In [2]:
# the REGISTERED RULE, applied in plain sight (TREE Q1.H2.E4, registered before scoring):
# a combination passes only if it scores >= 8 of 12 on the paper battery AND >= 8 of 12 on the
# held-out battery. The package supplies the sorted counts; the rule itself is these three lines.
from emotion_vectors.detection_report import combination_rows

instruct_rows = combination_rows(instruct_grids, context.layers)
print("top combinations (paper, heldout, format, layer, readout):")
for paper, held_out, fmt, layer, readout in instruct_rows[:10]:
    print(f"  paper {paper:2d}/12  heldout {held_out:2d}/12  {fmt:5s}  layer {layer:2d}  {readout}")

confirmed = [row for row in instruct_rows if row[0] >= PASS_BAR and row[1] >= PASS_BAR]
print(
    f"\nregistered rule - combinations passing BOTH batteries at >={PASS_BAR}/12: "
    f"{confirmed if confirmed else 'NONE'}"
)

top combinations (paper, heldout, format, layer, readout):
  paper  6/12  heldout  4/12  chat   layer 57  last
  paper  4/12  heldout  4/12  plain  layer 48  last
  paper  4/12  heldout  4/12  plain  layer 45  mean_all
  paper  4/12  heldout  4/12  plain  layer 27  mean_content
  paper  4/12  heldout  4/12  plain  layer 27  mean_all
  paper  4/12  heldout  4/12  chat   layer 51  mean_content
  paper  4/12  heldout  4/12  chat   layer 51  mean_all
  paper  4/12  heldout  3/12  plain  layer 54  mean_all
  paper  4/12  heldout  3/12  plain  layer 45  last
  paper  4/12  heldout  2/12  plain  layer 57  mean_content

registered rule - combinations passing BOTH batteries at >=8/12: NONE


In [3]:
# the same sweep on the base model, which has no chat template (plain format only),
# plus the assert that reproduces the archived bench run
base = context.model(BASE_LABEL)
base_grids = sweep_grids(base)
figure_base, stats_base = sweep_figure(
    base, base_grids, "extension of Anthropic Figure 2 (TREE Q1.H2.E2)"
)
figure_base.show()

base_rows = combination_rows(base_grids, context.layers)
print("top combinations, gemma-4-31b base (paper, heldout, format, layer, readout):")
for paper, held_out, fmt, layer, readout in base_rows[:5]:
    print(f"  paper {paper:2d}/12  heldout {held_out:2d}/12  {fmt:5s}  layer {layer:2d}  {readout}")

# REPRODUCTION CHECK (keep this assert): the base arm must still reproduce the archived
# best cell from archive/04_probe_sweep.ipynb, or the instrument has changed under us.
best = base_rows[0]
assert (best[0], best[1], best[3], best[4]) == (7, 4, 57, "last"), (
    f"base sweep no longer reproduces the archived best cell (archive/04_probe_sweep): {best}"
)
print(
    "reproduction check vs archive/04_probe_sweep - "
    "best cell 7/12 paper, 4/12 heldout at layer 57, last: OK"
)

base_confirmed = [row for row in base_rows if row[0] >= PASS_BAR and row[1] >= PASS_BAR]
print(
    f"\nregistered rule (gemma-4-31b base) - combinations passing BOTH batteries at "
    f">={PASS_BAR}/12: {base_confirmed if base_confirmed else 'NONE'}"
)

top combinations, gemma-4-31b base (paper, heldout, format, layer, readout):
  paper  7/12  heldout  4/12  plain  layer 57  last
  paper  5/12  heldout  5/12  plain  layer 57  mean_content
  paper  5/12  heldout  4/12  plain  layer 57  mean_all
  paper  4/12  heldout  4/12  plain  layer 33  mean_content
  paper  4/12  heldout  4/12  plain  layer 33  mean_all
reproduction check vs archive/04_probe_sweep - best cell 7/12 paper, 4/12 heldout at layer 57, last: OK

registered rule (gemma-4-31b base) - combinations passing BOTH batteries at >=8/12: NONE


<details><summary><b>How to read this figure</b></summary>

**What changed.** The same grid, computed for the base model (`gemma-4-31b`) from its own
probes and its own battery activations. The base model has no chat template, so only the plain
format exists: one panel pair instead of two. The instruct model above is the arm we care
about; this arm shows the failure predates chat formatting.

**The grading scale is the same**, and so is the verdict in the title, computed from this
cell's grids: a good result would be a bold cell confirmed in both panels, a bad result a grid
at chance, and the observed grid reaches a brightest cell of 7 on the selection panel that
falls to 4 beside it. That drop is the selection effect the held-out panel exists to expose.

**The printed check below the figure** confirms the best cell (7 of 12 on the paper battery, 4
of 12 held-out, at layer 57 with the last-token readout) exactly reproduces the archived bench
run (`archive/04_probe_sweep.ipynb`). It is an assert, so this notebook fails loudly rather
than quietly reporting a different number if the stored activations ever change.

</details>

**What section 1 establishes.** Under this readout convention, no combination of layer,
readout, and prompt format detects the implied emotion on either model: the rankings printed
above top out well short of the bar of 8 of 12 on both batteries, and most cells sit at the
chance level of about 3. This rules out "we picked the wrong layer or token position" as the
explanation, for this readout family.

**Live hypotheses after section 1** (each is a hypothesis, not a finding, and each is paired
with the experiment that decides it):

- *Probe-lineage hypothesis*: detection fails because the probes were built from a different,
  smaller model's stories. Deciding experiment: rebuild the probes from the probed model's own
  writing, section 3.
- *Probe-data-volume hypothesis*: detection fails because 16 stories per emotion is too thin an
  estimate of each direction. Deciding experiment: the 16-to-256 scale test, sections 4 and 5.
- *Confound hypothesis*: detection fails because a large non-emotional component dominates the
  activations. Deciding experiment: the paper's neutral projection, section 6.
- *Readout-convention hypothesis*: detection fails because the cosine is measured from the
  wrong baseline. Deciding experiment: the centered readout, TREE Q1.H2.E9, which is where the
  campaign in fact resolved (claim C4, header).

**Open question.** Nothing here separates "the probes carry no emotion information" from "the
probes carry it in a form this ranking read cannot see". Section 2 is the first evidence that
the second is closer to the truth.


## 2. Numerical intensity: does the reading move with what a number means? (paper Figure 3, the Tylenol plot)

Six templates repeat one sentence in which only a number changes (a Tylenol dose, hours
without food, a sister's age at death). If the probes track meaning rather than digits, the
cosine readings must move with what the number implies: more Tylenol should raise the
"afraid" reading and lower "calm". This is the second registered prediction of the first
experiment (TREE Q1.H2.E1, prediction P2), shown here for both models.


In [4]:
# load-call-show: cosine curves for all six templates at every layer, the figure with its
# layer slider, and the registered sign test printed at the registered layer 33
from emotion_vectors.detection_report import (
    REGISTERED_LAYER,
    intensity_curves,
    intensity_figure,
)

curves = intensity_curves(context)
figure_intensity, stats_intensity = intensity_figure(curves, context.layers)
figure_intensity.show()
print("\n".join(stats_intensity["lines"]))

# REPRODUCTION CHECK (keep this assert): E1 scored the base arm at 7 of 11 correct signs
# (plain format, last-token readout, layer 33). The printed record above must still say so.
assert stats_intensity["correct"][BASE_LABEL] == 7, (
    f"base arm should reproduce E1's 7/11 (got {stats_intensity['correct'][BASE_LABEL]}/11)"
)
print(f"reproduction check vs Q1.H2.E1 (plain last, layer {REGISTERED_LAYER}) - 7/11: OK")

gemma-4-31b-it - Spearman sign per registered (template, probe) direction:
  tylenol      afraid  registered +  rho +0.943  correct
  tylenol      calm    registered -  rho -1.000  correct
  fasting      afraid  registered +  rho +0.857  correct
  sister       sad     registered -  rho -0.829  correct
  sister       calm    registered +  rho +1.000  correct
  sister       happy   registered +  rho +0.600  correct
  dog_missing  sad     registered +  rho +1.000  correct
  runway       afraid  registered -  rho -0.943  correct
  runway       calm    registered +  rho +0.943  correct
  exam         happy   registered +  rho +1.000  correct
  exam         afraid  registered -  rho -0.600  correct
gemma-4-31b-it: 11/11 registered directions correct, read at the registered layer 33

gemma-4-31b (base) - Spearman sign per registered (template, probe) direction:
  tylenol      afraid  registered +  rho +0.771  correct
  tylenol      calm    registered -  rho -1.000  correct
  fasting      afra

<details><summary><b>How to read this figure</b></summary>

**What the axes are.** Each panel is one template: the same sentence, with only the number
changing. The x axis walks through the values that number takes (equally spaced, labelled with
the actual values). The y axis is the cosine between the sentence's activation and each of the
four tracked probes: afraid (red), calm (blue), happy (green), sad (orange). Solid lines are
the instruct model (`gemma-4-31b-it`) under its chat template; dashed lines are the base model
in plain format, which has no chat template. All curves are read at the last token, at the
layer the slider selects.

**Layer 33 (marked * on the slider) is the registered read**, and the sign scoring printed
below the figure always uses layer 33 whatever the slider is showing. Moving the slider
retitles the figure and every panel's grading line with that layer's own score, so a reader can
see how layer-specific the result is without mistaking another layer for the registered one.

**The grading scale.** The registered scoring (TREE Q1.H2.E1, prediction P2) is only the
*direction*: the sign of the Spearman rank correlation between the number and the cosine, for
11 named (template, probe) pairs. Each panel states its own registered directions in words
("afraid rises with the number") and its own score at the shown layer. A good result would be
all 11 signs correct; a bad result would be about 5.5 of 11, which is what coin flips give
under the generic-drift null below. The observed result at layer 33 sits at the good end: the
instruct model gets 11 of 11 and the base model 7 of 11, both printed above, and the base count
reproduces the archived result exactly (fasting and dog-missing invert; sister is mixed).

**Valid readings.** "The direction of movement is right far more often than chance on the
instruct model." "The base model is inconsistent." "The verdict is layer-specific: move the
slider and the counts change."

**Invalid readings.** "The lines are flat, so nothing happens" - the visible flatness is plot
autoscale over a tiny range, and the registered read is the sign, not the slope. "The steeper
the curve, the stronger the effect" - correlation size is not evidence here, for the reason in
the next block.

**Why the sign and not the size (confound check, TREE Q1.H2.E8,
`results/e8_template_diagnostic.json`).** A registered diagnostic ran three controls on the
instruct arm. First, a random-direction null: random directions in probe span reach a perfect
rank correlation with the number so often (null 95th percentile 1.00 on 5 of 6 panels) that
correlation size is not evidence; the last-token activation drifts near-monotonically with the
number regardless of emotional content, and the activation norm itself tracks the number
(absolute rho 0.54 to 0.83 on 5 of 6 panels). Second, amplitude: our per-curve cosine
excursions run from 0.0015 to 0.0162 against the paper's roughly 0.16 full range, so 10x to
100x smaller; the visible slopes here are plot autoscale. Third, the neutral-PC projection
changes neither result. What survives is exactly the registered sign read: under the
generic-drift null each named direction is a coin flip, so 11 of 11 correct signs has
probability about 2 to the power -11.

</details>

**What section 2 establishes.** The probes carry a real, direction-consistent trace of what
the numbers imply: the instruct model moves all 11 registered directions the right way, the
base model 7 of 11 (both printed above). But the effect is a tiny tilt riding on a much larger
generic drift, so it supports "a coarse emotional signal exists" and nothing stronger. This is
the one positive read of the uncentered campaign.

**Live hypotheses after section 2.**

- *Coarse-signal hypothesis*: the probes encode valence (good-vs-bad tone) reliably and the
  specific emotion only weakly, which would explain a passing sign test beside a failing
  ranking test. Deciding experiment: a valence-only read of the same activations, reported in
  the geometry notebook (`notebooks/02_circumplex_geometry.ipynb`) rather than here.
- *Readout-convention hypothesis* (carried forward from section 1): the sign test works
  because it is invariant to the shared offset that the ranking read is dominated by. Deciding
  experiment: remove that offset from the activations and re-run the ranking read, TREE
  Q1.H2.E9 (claim C4).

**Open question.** The sign test and the ranking test disagree on the same activations. Which
of the two is measuring the thing we care about is not settled by this section alone.


## 3. Probes from the model's own writing (gemma-4-31b-it only)

Section 1's probes were built from stories written by a different, smaller model. This
section rebuilds them from stories
([`abotresol/emotion-stories-gemma-4-31b-it`](https://huggingface.co/datasets/abotresol/emotion-stories-gemma-4-31b-it))
and dialogues
([`abotresol/emotion-dialogues-gemma-4-31b-it`](https://huggingface.co/datasets/abotresol/emotion-dialogues-gemma-4-31b-it))
that the probed model wrote itself, testing the probe-lineage hypothesis from section 1.

*Instruct-only by design: the self-generated story and dialogue corpora exist only for
`gemma-4-31b-it`. The base arm has no equivalent here (base-model dialogue probes live in
`archive/05_dialogue_probes.ipynb`).*


In [5]:
# load-call-show: the three probe sets, how far the self-generated directions sit from the
# corpus directions, and the leakage quality check on the model's own texts
from emotion_vectors.detection_report import (
    direction_similarity,
    leakage_report,
    load_probe_sets,
)

probe_sets = load_probe_sets(context)
similarity = direction_similarity(probe_sets, context.layers)
leakage = leakage_report()
print("\n".join(similarity["lines"][:3]))
print("\n".join(leakage["lines"]))
print("\n".join(similarity["lines"][3:]))

model: gemma-4-31b-it (all probe sets and battery activations)
probe-direction cosine vs 4B-corpus probes (self-story, layer 33): mean 0.220, min -0.027
probe-direction cosine vs 4B-corpus probes (dialogue, layer 33): mean 0.093, min -0.201
leakage QC (self-story corpus): 42/3072 texts name their emotion (1.4%)
leakage QC (dialogue corpus): 0/192 texts name their emotion (0.0%)

mean probe-direction cosine vs 4B-corpus probes, all layers:
layer  self-story  dialogue
    0       0.193     0.029
    3       0.277     0.073
    6       0.299     0.124
    9       0.297     0.141
   12       0.262     0.076
   15       0.278     0.065
   18       0.304     0.120
   21       0.327     0.173
   24       0.251     0.204
   27       0.187     0.084
   30       0.216     0.106
   33       0.220     0.093  *
   36       0.234     0.095
   39       0.235     0.073
   42       0.296     0.084
   45       0.305     0.080
   48       0.269     0.065
   51       0.244     0.033
   54       0.174     

In [6]:
# the REGISTERED RULE again, applied in plain sight to every (probe set, format, layer,
# readout): pass = >= 8 of 12 on the paper battery AND >= 8 of 12 on the held-out battery.
# battery_counts is the shared scorer; these sets center on their own 12 probes (see the
# package docstring), because they never extracted the full 171-word vocabulary.
from emotion_vectors.detection_report import READOUTS, battery_counts, rank_table_lines

rows, confirmed = [], []
for probe_set_name, probes in probe_sets.items():
    for fmt in instruct.formats:
        for layer_pos, layer in enumerate(context.layers):
            for readout in READOUTS:
                paper, held_out = battery_counts(
                    instruct, probes[:, layer_pos, :], fmt, readout, layer_pos
                )
                rows.append((paper, held_out, probe_set_name, fmt, layer, readout))
                if paper >= PASS_BAR and held_out >= PASS_BAR:
                    confirmed.append(rows[-1])
rows.sort(reverse=True)
print("\n".join(rank_table_lines(rows)))
print(
    f"\nregistered rule - combinations passing BOTH batteries at >={PASS_BAR}/12: "
    f"{confirmed if confirmed else 'NONE'}"
)

probe sets scored on gemma-4-31b-it battery activations:
  corpus     = probes from gemma-4-4B stories (section 1's probes)
  self-story = probes from stories the probed model wrote itself
  dialogue   = probes from dialogues the probed model wrote itself
 paper  heldout  set         fmt    layer readout
   7/12     6/12  self-story  chat    57  mean_content
   7/12     6/12  self-story  chat    57  mean_all
   6/12     5/12  self-story  chat    57  last
   6/12     5/12  dialogue    chat    57  mean_all
   6/12     4/12  dialogue    chat    57  last
   5/12     4/12  corpus      chat    57  last
   4/12     5/12  self-story  chat    54  last
   4/12     4/12  dialogue    plain   24  mean_all
   4/12     4/12  dialogue    chat    51  mean_content
   4/12     4/12  corpus      plain   27  mean_content
   4/12     4/12  corpus      plain   27  mean_all
   4/12     4/12  corpus      chat    48  last

registered rule - combinations passing BOTH batteries at >=8/12: NONE


<details><summary><b>How to read this output</b></summary>

**What the table is.** Each row is one probe set scored on the same two batteries as section 1,
at one (format, layer, readout) combination, best first, with at most four rows per probe set so
one set cannot crowd out the others. "corpus" probes come from the published gemma-4-4B
stories (section 1's probes); "self-story" and "dialogue" probes come from stories and
dialogues the probed model wrote itself. The two count columns are the paper battery and the
held-out battery, both out of 12.

**The quality check above it.** The leakage lines are the check that the model's own texts do
not simply name the target emotion, which would make the probes read a word rather than a
state: self-story 1.4% of texts, dialogue 0.0% (printed above). The cosine table above that
says how far the self-generated probe directions sit from the corpus probe directions at every
layer, so a reader can see these are genuinely different probe sets and not a relabelling.

**The grading scale.** A good result would be a row with 8 or more in both count columns. A bad
result would be rows at 3 and 3, the chance level. The observed best sits between them and
below the bar: self-generated probes give the campaign's best cells, 7 of 12 paper with 6 of 12
held-out (printed above), and nothing reaches 8 on both.

**Valid readings.** "Matching the generator to the probed model helps." "It is not enough."
**Invalid reading.** "7 of 12 is close to 8, so it nearly passed" - the bar is 8 on BOTH
batteries, and this row's held-out count is 6, so it misses on the column that was not used to
select it.

</details>

**What section 3 establishes.** Rebuilding the probes from the model's own stories and
dialogues improves scores (the campaign best moves to 7 of 12 with 6 of 12 held-out, printed
above) and still fails the registered rule. The probe-lineage hypothesis from section 1 is
therefore weakened, not refuted: matching the generator to the probed model helps, but does not
rescue detection under this readout.

**Live hypotheses after section 3.**

- *Partial-lineage hypothesis*: lineage matters but this notebook only tried two generators, so
  a stronger generator might clear the bar. Deciding experiment: the four-lineage head-to-head,
  `notebooks/07_generator_lineages.ipynb` (TREE Q1.H2.E10-E12), which found that it does under
  the centered readout.
- *Probe-data-volume hypothesis* (carried forward): the self-generated corpus is also small.
  Deciding experiment: section 4.

**Open question.** The self-generated probe directions agree with the corpus probes at a mean
cosine of only 0.220 at layer 33 (printed above), yet both score about the same. What the two
sets share that the battery read is picking up is not identified here.


## 4. The scale test: does more probe data help? (gemma-4-31b-it only)

The probe-data-volume hypothesis, tested directly: build the probes from 16, 64, 128, and 256
stories per emotion and score the battery at each size.

*Instruct-only by design: the 16-to-256-stories-per-emotion scale corpus was generated by and
extracted from `gemma-4-31b-it`; no base-arm equivalent exists.*


In [7]:
# load-call-show: the scale bundle, the best battery score at each corpus size, and the curve
from emotion_vectors.detection_report import load_scale_corpus, scale_figure, scale_rows

scale = load_scale_corpus(context.layers)
print(
    f"model: {IT_LABEL} | probe means {scale.means.shape} "
    f"[corpus sizes, emotions, layers, d_model] | corpus sizes {scale.n_buckets} "
    f"| formats {instruct.formats} | {len(scale.layers)} layers"
)

rows_by_size = scale_rows(instruct, scale)
figure_scale, stats_scale = scale_figure(rows_by_size)
figure_scale.show()
print("\n".join(stats_scale["lines"]))

model: gemma-4-31b-it | probe means (4, 12, 20, 5376) [corpus sizes, emotions, layers, d_model] | corpus sizes [16, 64, 128, 256] | formats ['plain', 'chat'] | 20 layers


n= 16: best paper 5/12, best heldout 6/12, best joint config chat format, layer 39, last token readout (5/12 paper, 5/12 held-out)
n= 64: best paper 5/12, best heldout 4/12, best joint config plain format, layer 42, mean of content tokens readout (4/12 paper, 4/12 held-out)
n=128: best paper 5/12, best heldout 5/12, best joint config chat format, layer 57, last token readout (5/12 paper, 5/12 held-out)
n=256: best paper 4/12, best heldout 5/12, best joint config plain format, layer 6, mean of all tokens readout (4/12 paper, 4/12 held-out)
peak joint score over all corpus sizes: 5/12, bar 8/12


<details><summary><b>How to read this figure</b></summary>

**What the axes are.** The x axis is how many stories per emotion the probes were built from,
16 to 256, on a log scale. The y axis is the battery score: scenarios, out of 12, whose true
emotion ranked in the top 3.

**What the three lines are.** The blue line takes the best score on the paper battery over
every layer, readout, and format at that corpus size. The orange dashed line does the same on
the held-out battery. Those two are maximised *separately*, so they can come from different
combinations, and a point where both are high does not mean any single combination scored high
on both. The red line is the one the registered rule actually grades: the single combination
whose worse battery is highest.

**The grading scale.** The green dotted line at 8 of 12 is the registered pass bar; the grey
dotted line at 3 of 12 is chance. A good result would be the red line rising to touch the green
line as the corpus grows. A bad result would be all three lines flat at the grey line. The
observed pattern sits between: flat, roughly 4 to 6 of 12, never approaching the bar, with the
per-size numbers printed below the figure.

**Valid reading.** "Growing the probe corpus 16x buys nothing."
**Invalid reading.** "The held-out line starts above the paper line at 16 stories, so the small
corpus generalises better" - both are maxima over about 120 combinations each, so the gap is
selection noise, not a difference in generalisation.

</details>

**What section 4 establishes.** The curves stay flat: the printed per-size bests range from 4
to 6 of 12 and never approach 8. The probe-data-volume hypothesis from section 1 is refuted for
this readout: "not enough probe data" is not why detection fails.

**Live hypothesis after section 4.** *Noise hypothesis*: the probes might still be noisy
estimates in a way the score curve cannot show, because a flat score is also what you would see
if the score were pinned by something else entirely. Deciding experiment: measure whether the
probe directions themselves have converged, section 5.

**Open question.** A flat curve is consistent with both "the probes are already as good as they
get" and "the read is broken regardless of the probes". Section 5 separates those.


## 5. Probe-direction convergence: are the probes stable, or noisy? (gemma-4-31b-it only)

The noise hypothesis from section 4, tested directly: do the probe directions built from a
small corpus already agree with the directions built from the full corpus?

*Instruct-only by design: convergence is measured on the `gemma-4-31b-it` scale corpus from
section 4; no base-arm equivalent exists.*


In [8]:
# load-call-show: how close each corpus size's probe directions come to the full-corpus
# directions, with section 3's cross-corpus cosine drawn as the "different probe set" comparator
from emotion_vectors.detection_report import convergence_figure

figure_convergence, stats_convergence = convergence_figure(
    scale, similarity["per_layer"]["self-story"]
)
figure_convergence.show()
print("\n".join(stats_convergence["lines"]))

n= 16 vs n=256: mean contrast-direction cosine 0.944 (min 0.894)
n= 64 vs n=256: mean contrast-direction cosine 0.991 (min 0.983)
n=128 vs n=256: mean contrast-direction cosine 0.997 (min 0.993)


<details><summary><b>How to read this figure</b></summary>

**What the axes are.** The x axis is again stories per emotion, on a log scale, and stops at
128 because 256 is the reference every point is compared against. The y axis is the cosine
between that corpus size's probe directions and the full 256-story directions, averaged over
the 12 emotions. Each set's own mean is removed first, so this compares what makes each emotion
different from the others, not the shared offset every probe carries.

**The grading scale.** Three reference marks are drawn in the plot. The green dotted line at
1.0 is identical directions, the strength anchor. The grey dotted line at 0 is unrelated
directions, the failure anchor. The grey dashed line between them is a measured comparator: how
far a genuinely *different* probe set sits, taken from section 3 (self-generated stories against
the gemma-4-4B corpus probes) at whatever layer the slider is showing. A good result would be
the blue line hugging 1.0; a bad result would be the blue line down near the comparator, which
would mean a 16-story corpus builds directions as different from the full corpus as a whole
other corpus does. The observed line sits at the good end from the start and reaches cosine
0.997 by 128 stories (printed below the figure, at layer 33).

**The slider** redraws the same curve and moves the comparator to that layer's own value, and
retitles with that layer's verdict. The printed numbers report layer 33, which is where the
campaign record reports it.

**Valid reading.** "The directions have converged; more stories would move them very little."
**Invalid reading.** "Converged directions must be correct directions" - this figure says the
estimate is stable, not that it points anywhere useful. Section 4's flat score curve is what
says the stable direction still does not rank the right emotion into the top 3.

</details>

**What section 5 establishes.** The probes are not noisy estimates that more data would fix: by
128 stories they agree with the 256-story reference at cosine 0.997 (printed above), far above
the cross-corpus comparator drawn in the figure. Combined with section 4's flat score curve,
the noise hypothesis is refuted. Whatever the probes point at is stable; under this readout it
is simply not enough to rank the right emotion into the top 3.

**Live hypothesis after section 5.** *Confound hypothesis* (carried forward from section 1):
the stable direction is dominated by a large non-emotional component. Deciding experiment: the
paper's neutral projection, section 6.

**Open question.** Convergence at cosine 0.997 with a battery score stuck near chance means the
failure is in the read, not the estimate. That points at the readout convention, which is where
the resolution landed.


## 6. Neutral projection: does the paper's confound-removal step rescue detection? (gemma-4-31b-it only)

The confound hypothesis, tested with the source paper's own repair: find the directions that
vary while the model reads emotionally neutral text and subtract them out of the probes.

*Instruct-only by design: the neutral transcripts used for the confound projection
([`abotresol/neutral-transcripts-gemma-4-31b-it`](https://huggingface.co/datasets/abotresol/neutral-transcripts-gemma-4-31b-it))
were generated by `gemma-4-31b-it`; no base-arm equivalent exists.*


In [9]:
# load-call-show: apply the neutral projection to the full-corpus probes, score before and
# after, then apply the registered rule to whichever probe set did best
from emotion_vectors.detection_report import (
    best_scores,
    describe_configuration,
    projection_probes,
    projection_table,
)

unprojected, projected, components_removed = projection_probes(scale)
table = projection_table(
    instruct,
    {"unprojected": unprojected, "projected": projected},
    context.layers,
    components_removed,
)
print("\n".join(table["lines"]))

_, configuration_unprojected = best_scores(instruct, unprojected)
_, configuration_projected = best_scores(instruct, projected)
print(f"\nbest joint (unprojected): {describe_configuration(configuration_unprojected)}")
print(f"best joint (projected):   {describe_configuration(configuration_projected)}")

# the REGISTERED RULE once more, on either probe set: >= 8 of 12 on BOTH batteries
passing = [
    configuration
    for configuration in (configuration_unprojected, configuration_projected)
    if configuration[3] >= PASS_BAR and configuration[4] >= PASS_BAR
]
print(f"registered rule, either probe set: {'PASS ' + str(passing) if passing else 'NONE pass'}")

model: gemma-4-31b-it
neutral PCs removed per layer (50% of the neutral variance): {0: 4, 3: 6, 6: 7, 9: 7, 12: 3, 15: 4, 18: 4, 21: 4, 24: 5, 27: 8, 30: 7, 33: 5, 36: 5, 39: 4, 42: 4, 45: 3, 48: 3, 51: 3, 54: 4, 57: 4}

probes       fmt    layer readout        paper  heldout
unprojected  plain     57 mean_content     3/12      3/12
unprojected  chat      57 last             4/12      5/12
unprojected  chat      57 mean_content     4/12      3/12
projected    plain     57 mean_content     4/12      4/12
projected    chat      39 last             5/12      5/12
projected    chat      39 mean_content     4/12      4/12
projected    chat      57 last             4/12      6/12
projected    chat      57 mean_content     5/12      4/12

best joint (unprojected): plain format, layer 6, mean of all tokens readout (4/12 paper, 4/12 held-out)
best joint (projected):   chat format, layer 18, mean of all tokens readout (5/12 paper, 5/12 held-out)
registered rule, either probe set: NONE pass


<details><summary><b>How to read this output</b></summary>

**What the step is.** The paper's confound-removal: find the directions that vary most in the
model's activations while it reads emotionally *neutral* text (via PCA, principal component
analysis), and subtract those directions out of the probes; what remains should be more purely
emotional. The first printed line shows how many neutral components are removed at each layer,
enough to cover half the neutral variance.

**What the table is.** The battery scored with unprojected against projected probes at the
full 256-story corpus size, at three layers and two readouts, showing only the rows whose two
counts together clear 8 (plus layer 57 with the content-mean readout as a fixed reference
point, so the table always has an anchor row). The last two printed lines give the best single
combination for each probe set, chosen by its worse battery, which is what the registered rule
grades.

**The grading scale.** A good result would be a best-joint line reading 8 or more on both
batteries. A bad result would be projection changing nothing. The observed result sits between:
projection adds roughly one scenario in several cells and moves the best joint configuration
from 4 and 4 to 5 and 5 (printed above), against a bar of 8. The geometric effect of the same
projection is in the geometry notebook (`notebooks/02_circumplex_geometry.ipynb`), section 7.

**Valid reading.** "The confound projection helps a little."
**Invalid reading.** "Projection improved the best cell, so the confound explanation is
confirmed" - a one-scenario move is within what a single re-ranking produces, and the two best
configurations sit at different layers and formats, so they are not the same measurement
improved.

</details>

**What section 6 establishes.** The paper's own confound-removal step helps slightly and does
not rescue detection: the best joint configuration reaches 5 of 12 on both batteries against a
bar of 8 (printed above). The confound hypothesis from section 1 is weakened: removing the
neutral-text directions is not sufficient. This closes the last registered repair inside this
readout family.

**Open question.** One concrete anomaly survives the projection: a dominant non-affective
component in the instruct model that the neutral PCs do not capture. It is described in the
geometry notebook and remains unexplained.


## 7. Verdict and the standing explanation

Every methodological repair the source papers and our own analysis could motivate has been
tried under pre-registered rules: instruct model, chat formatting, all layers and readouts,
dialogue-form probes, generator-matched corpora, a 16x scale-up, and the paper's confound
projection. Best observed configuration: 7 of 12 on the paper battery with 6 of 12 held-out
(section 3's printed table), against a bar of 8 and 8.

Where the campaign landed (gate results, 2026-07-22/23):

- This notebook's null claim (C3) graduated WEAKENED at its falsify gate
  (`results/falsify_c3_scorecard.json`). Its null component stands for the uncentered readout
  family scored throughout this notebook. Its coarse-signal component rests on diagonal-gap
  evidence rather than top-3 counts.
- The campaign's resolution is the centered-readout claim (C4), which SURVIVED its own gate
  (`results/falsify_c4_scorecard.json`). Under a centered-cosine readout (the scenario-set
  mean is subtracted from each scenario activation before taking cosines), story-derived
  probes DO pass the registered dual-battery bar on both models. The sweep-wide,
  selection-adjusted permutation null produced a passing layer in zero of 10,000 draws for
  the self-generated lineage and 2 of 10,000 for the DeepSeek lineage.
- The earlier "the two models differ" explanation is therefore retired in favor of a
  readout-convention difference. One concrete anomaly stays open: the dominant non-affective
  component in the instruct model that survives neutral projection.

**What the seven experiments together establish.** Detection fails under the uncentered
readout for reasons that are not the layer, not the token position, not the prompt format, not
the probe generator, not the corpus size, not sampling noise in the probes, and not the
neutral-text confound. Each of those was a live hypothesis and each was tested and closed in
the sections above. What was left standing, by elimination, was the readout convention itself,
and that is what the centered read then changed.

**Open questions carried out of this notebook.**

- The dominant non-affective component in the instruct model that survives neutral projection
  is unidentified.
- The sign test (section 2) and the ranking test (sections 1, 3, 4, 6) disagree on the same
  activations; which one measures the construct is not settled here.
- Why lineage changes how *broadly* probes pass under the centered readout, when it barely
  changes anything under the uncentered one, is open (`notebooks/07_generator_lineages.ipynb`).

TREE.md nodes Q1.H2.E1 through E12 carry every verdict with evidence paths. The lineage story
(which corpora make the best probes) is `notebooks/07_generator_lineages.ipynb`.


## 8. Do the probes predict what the model would rather do? (paper Figure 4, left half)

Sections 1 to 7 asked one question: given a scenario that implies an emotion, does the right probe
win? The answer was no, under the uncentered readout. This section and the next ask a different
question with the same vectors, the question the source paper's Figure 4 asks, and it is a question
about **behavior** rather than about scenario detection. That is why they sit after the verdict
rather than inside it: they are not an eighth attempt to make detection work, they are a separate
replication, folded in here from the paper-parity notebook so each comparison lives beside the
result it grades. The inventory of every source-paper figure and where it now lives is the front
matter of [notebook 02](02_circumplex_geometry.ipynb).

**What was collected.** The model is shown two activities and asked which it would rather do, over
every ordered pair of an activity list we wrote ourselves before any measurement (the paper's list
is unpublished; ours follows its named categories). A **Bradley-Terry** fit turns those pairwise
choices into one number per activity, an **Elo rating**, so that a gap in Elo predicts a win
probability. Then, separately, each emotion probe is projected onto the model's activation while it
reads an activity, and that projection is correlated with the activity's Elo. If the emotion vectors
carry preference-relevant content, some probe should track the ratings.

**Two prompt formats, because the first one was a dead instrument.** The paper's exact plain-text
format is out of distribution for an instruction-tuned model, and on it the model reveals no
coherent preferences at all. The chat-template arm is the same experiment through the model's own
conversation format. Both are shown below, in that order, so the format lesson stays visible instead
of being quietly dropped.

**Registered reads** (declared in `scripts/score_preferences.py` before the collection landed):
P1, the categories a well-behaved model should like out-rate the ones it should not; P2, the
strongest probe reaches the registered correlation bar with permutation-significant valence
organization. Both verdicts are printed by the cell, read from the scorer's own output.

Everything here comes from the padding-fixed collections (TREE Q1.H3.E4); the compromised originals
stay in the results history and on the dataset card.

In [10]:
# section 8 is load-call-show: the paper's Figure 4 left half for each collected arm, with a
# slider over every scored probe layer; the printed record carries the registered P1/P2 verdicts
from emotion_vectors.detection_report import load_preference_arms, preference_figure

for preference_arm in load_preference_arms():
    preference_fig, preference_stats = preference_figure(preference_arm)
    preference_fig.show()
    for preference_line in preference_stats["lines"]:
        print(preference_line)

[plain (paper's exact format)] P1 PASS (positive 1002 vs negative 1002); P2 FAIL (max |r|=0.4141 bar 0.5, organization r=-0.2293, perm p=0.0034)
  best layer 36, best probe 'amazed', evidence results/preferences_it_fixed/scores.json


[chat template] P1 PASS (positive 1727 vs negative -578); P2 PASS (max |r|=0.7013 bar 0.5, organization r=0.4189, perm p=0.0)
  best layer 33, best probe 'outraged', evidence results/preferences_it_chat_fixed/scores.json


<details><summary><b>How to read these two panels</b></summary>

**What the axes are.** Left panel: one bar per emotion probe, sorted, its height the Pearson
correlation between that probe's activation while the model reads an activity and that activity's
Elo rating. Right panel: one dot per activity, its horizontal position the strongest probe's
activation and its vertical position the Elo, colored by what kind of activity it is. The slider
moves both panels to another scored probe layer and re-titles the figure with that layer's own
verdict, so no single layer is privileged.

**The grading scale, drawn in the plot.** A bad result is the flat line at zero: a probe that says
nothing about preference. The registered bar is the dashed line the figure labels, mirrored below zero
because the statistic is an absolute value. A strong result is the shaded band, the range the source
paper reports. The observed pattern sits between them: the chat arm clears the registered bar and
lands just under the paper's band, and the plain arm does not clear the bar at any layer.

**Valid readings.**
- "In the chat arm, the bars run from strongly negative to strongly positive and the extremes are
  the emotions valence would predict, so preference is organized by valence and not by noise."
- "The right panel separates cleanly by category: the activities a well-behaved model should refuse
  sit at the bottom of the Elo scale. That is the P1 verdict, printed under the figure."
- "The plain arm's Elo spread is tiny compared with the chat arm's, which is what a model with no
  coherent preferences looks like."

**Invalid readings.**
- "The best probe is the emotion the model feels about these activities." The bar chart is a
  correlation ranking over many probes; the top one is a selection, and its identity moves with the
  layer. Read the shape of the ranking, not its winner.
- "Our Elo numbers are lower than the paper's." Elo here is anchored at a fixed mean by
  construction and the paper's anchoring convention is unpublished. Only gaps and rankings compare.
- "This shows the vectors cause the preference." Nothing here is causal. Section 9 is the causal
  test.

</details>

**Instrument sensitivity, before the headline is quoted anywhere.** The probes themselves were
re-extracted after the padding fix, so the correlation above can be computed against two different
probe instruments. If the headline moved a lot between them, it would be a property of the
instrument rather than of the model. The cell below scores the same chat-arm activations against
both probe sets, layer by layer.

In [11]:
# load-call-show: the same chat-arm activations scored against the pre-fix and post-fix probe
# sets, so section 8's headline correlation is stated as a range over instruments
from emotion_vectors.detection_report import instrument_sensitivity_stats

for instrument_line in instrument_sensitivity_stats()["lines"]:
    print(instrument_line)

probe-Elo max |r| per layer, chat arm, by probe instrument (registered bar 0.5):
 layer  pre-fix probes  post-fix probes
    24          0.5948           0.5970
    30          0.6368           0.6223
    33          0.7013           0.6448
    36          0.6514           0.5658
sources: results/preferences_it_chat_fixed/scores.json (pre-fix probes) and scores_postfix_probes.json (post-fix probes)
read: every layer clears the 0.5 bar under both instruments; the fix costs |r| 0.7013 to 0.6448 at the best layer (33) but does not change any pass/fail verdict


## 9. Does steering with an emotion vector move those preferences? (paper Figure 4, right half)

Section 8 is correlational: probes that predict preference could be reading the same thing
preference reads, without carrying it. The causal test is to add an emotion vector to the residual
stream while the model makes its choices, and ask whether the preferences move, and whether they
move in the direction that emotion's probe predicted.

**What was collected.** The full chat-format pair pass was re-run under steering with each battery
emotion, at two doses spanning the range that leaves the model's choices coherent. The measured
quantity is deliberately a **redistribution**: the mean Elo change of the positive-category
activities. A uniform shift of every activity is invisible to pairwise choices by construction, so a
global mean would be identically zero no matter how strong the steering was. That degeneracy was
found and the statistic amended before scoring, and it is recorded in the tree.

**Registered reads** (TREE Q1.H3.E2): P1, steering leaves the choices coherent; P2, the size of the
effect correlates with the correlational profile from section 8 at the registered bar; P3, the
valence-sign test, each emotion moves preferences in the direction its valence predicts. All three
verdicts are printed by the cell.

In [12]:
# section 9 is load-call-show: the paper's Figure 4 right half, both steering doses on one
# scatter; the printed record carries the registered P1/P2/P3 verdicts per dose
from emotion_vectors.detection_report import load_steering_arms, steering_figure

steering_fig, steering_stats = steering_figure(load_steering_arms())
steering_fig.show()
for steering_line in steering_stats["lines"]:
    print(steering_line)

[alpha=2] P1 PASS; P2 FAIL (r=+0.322, p=0.307); P3 PASS (11/12)
[alpha=8] P1 PASS; P2 FAIL (r=+0.228, p=0.475); P3 PASS (10/12)
evidence: results/steering_it_fixed/scores.json, results/steering_it_a8_fixed/scores.json


<details><summary><b>How to read this scatter</b></summary>

**What the axes are.** One dot per steered emotion, at each of the two doses (the legend names which
dose is which). The horizontal axis is how well that emotion's probe *predicted* preference without
any steering, which is section 8's per-probe correlation. The vertical axis is how far steering with
that same vector actually *moved* preference, as the mean Elo change of the positive-category
activities.

**The grading scale, drawn in the plot.** The solid line at zero is the failure anchor: steering
that redistributes nothing. The two dashed lines are the strength anchors, the mean shifts the
source paper reports for its most positive and most negative steering vectors. The paper's claim is
also about the *shape* of this cloud: it reports a strong positive slope between prediction and
effect. Ours is quoted in the legend for each dose, against the registered bar in the title. The
observed pattern: the vertical spread grows with dose and stays inside the paper's anchors at the
larger dose, while the slope stays well below the bar.

**Valid readings.**
- "Positive-valence emotions sit above the zero line and negative-valence emotions below it, which
  is the valence-sign test passing, and it is printed as P3 under the figure."
- "The larger dose moves preferences further than the smaller one, so the effect is dose-responsive
  rather than an artifact of one arbitrary strength."
- "The cloud is not a line. Knowing how well a probe predicted preference does not tell you how much
  steering with it will move preference."

**Invalid readings.**
- "Steering does nothing." It does: the direction is right at both doses and the magnitude tracks
  the dose. What fails is the fine-grained coupling, not the effect.
- "This contradicts the paper." The direction reproduces. The coupling does not, and the range our
  battery emotions span on the horizontal axis is much narrower than the paper's, which is one
  candidate explanation rather than a proven one.
- "The effect sizes are comparable to the paper's." Only the larger dose reaches the paper's
  anchors, and only for the strongest emotion. Read each dot against the dashed lines, not against
  the other dots.

</details>

**What sections 8 and 9 establish.** On the model's own chat format, the emotion vectors do carry
preference-relevant content: probes predict revealed preference above the registered bar, and
steering with them moves preference in the valence-correct direction at both doses, dose-responsively
and without breaking the coherence of the choices. That is the source paper's Figure 4 reproducing in
direction and in rank. What does not reproduce is the tight coupling between prediction and effect
that the paper reports, and the plain-text format that the paper used remains a dead instrument on an
instruction-tuned model. In the parity inventory this is the entry "Figure 4: diverged", and the
divergence is specific and small enough to name.

Read together with sections 1 to 7, the pattern is the one this project keeps meeting: the
representation is present, and the readout is weak. Scenario detection fails under the uncentered
readout while behavior prediction and causal steering, which read the same vectors a different way,
both work.

**Live hypotheses, and the experiment that would decide each.**

- *Hypothesis (range, not weakness):* the prediction-effect coupling looks flat because our
  battery emotions span a narrow slice of the horizontal axis. *Deciding experiment:* steer with
  emotions chosen to span the full valence range of the probe bank, including the extreme probes
  section 8's bar chart identifies, and re-fit the same correlation.
- *Hypothesis (readout position):* steering is applied at every token position at one layer, which
  is a blunter intervention than the paper's. *Deciding experiment:* steer at the choice token only,
  and sweep the layer, scoring the same P1 to P3 reads.
- *Hypothesis (expression, not preference):* steering changes what the model *says* rather than what
  it prefers. *Deciding experiment:* the delta-log-probability variant of the paper's steering
  figures, which measures the expressed token distribution instead of the fitted rating. It is the
  top queued item in `notes/plot_parity.md`.

**Open question.** The activity list is ours, not the paper's, so the Elo scale and the category
split are not literally comparable to theirs. Whether the coupling gap survives on the paper's own
activities cannot be answered until that list is published.